# 01 — Building the dataset and auditing the labels

**Purpose:** download JARVIS-DFT, keep every material with a bulk modulus value, label each one by
whether ALIGNN's pretrained model was trained on it, and count physically impossible bulk modulus values.

**Produces:** Table 1 of the paper, and `data/full_bulk_modulus_audit.csv`.

**Runs on:** a laptop. The first run downloads about 1 GB from JARVIS and caches it, so later runs are fast.

**Note:** JARVIS-DFT keeps growing. These outputs come from a download in September 2026; a fresh
download may give slightly different counts.

## 1. Download JARVIS-DFT

`data("dft_3d")` fetches the full 3D database. Each entry is a dictionary describing one material.

**Expect:** 93,902 materials (September 2026).

In [20]:
import json
from jarvis.db.figshare import data

d = data("dft_3d")
print (f'materials in JARVIS: {len(d)}')

Obtaining 3D dataset 94k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815699
Loading the zipfile...
Loading completed.
materials in JARVIS: 93902


## 2. Keep materials with a bulk modulus

JARVIS stores missing values as the text `"na"`. `pd.to_numeric(..., errors="coerce")` turns that text into
a real blank (`NaN`), and `.notna()` then keeps only rows that have a number.

`bulk_modulus_kv` is the Voigt bulk modulus in GPa, the property the pretrained model predicts.

**Expect:** 32,433 materials.

In [21]:
import pandas as pd
rows = []
for entry in d:
    rows.append({
        "jid":entry.get("jid"),
        "formula": entry.get("formula"),
        "bulk_modulus_kv": entry.get("bulk_modulus_kv","na"),
        "optb88vdw_bandgap": entry.get("optb88vdw_bandgap","na"),
    })
full = pd.DataFrame(rows)
full["bulk_modulus_kv"] = pd.to_numeric(full["bulk_modulus_kv"], errors="coerce")
full["optb88vdw_bandgap"] = pd.to_numeric(full["optb88vdw_bandgap"], errors="coerce")

full = full[full["bulk_modulus_kv"].notna()]
print(f"Materials with a bulk modulus value: {len(full)}")

Materials with a bulk modulus value: 32433


## 3. Get the model's training, validation and test lists

Every pretrained ALIGNN model is distributed with a file, `ids_train_val_test.json`, listing exactly which
JARVIS materials went into each split. It sits inside the model's zip on Figshare, so we read it straight
from there.

This file is what makes the whole study possible: without it, there is no way to know which materials
the model has already seen.

**Expect:** 15,744 / 1,968 / 1,968 identifiers.

In [ ]:
import requests, zipfile, io

# The split file ships inside the pretrained model's zip on Figshare
r = requests.get("https://ndownloader.figshare.com/files/67163351", timeout=300)
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    hit = [n for n in z.namelist() if n.endswith("ids_train_val_test.json")][0]
    with z.open(hit) as f:
        ids = json.load(f)

train_ids = set(ids["id_train"])
val_ids = set(ids["id_val"])
test_ids = set(ids["id_test"])

## 4. Classify every material

Each material goes into one of four groups. Anything in none of the three splits is `not_in_split` —
called *unseen* in the paper.

`.apply(categories)` runs the function once for every value in the `jid` column.

**Expect:** train 15,744 · val 1,968 · test 1,968 · not_in_split 12,753. The first three must match the
split file exactly, which confirms the matching worked.

In [ ]:
def categories(jid):
    if jid in train_ids:
        return "train"
    if jid in val_ids:
        return"val"
    if jid in test_ids:
        return"test"
    return "not_in_split"
full["split"] = full["jid"].apply(categories)
print(full["split"].value_counts())


split
train           15744
not_in_split    12753
test             1968
val              1968
Name: count, dtype: int64


## 5. Physically impossible values (Table 1)

A bulk modulus measures how hard a material is to compress. For a stable crystal it must be positive, so
a value of zero or below means the DFT calculation failed.

**Expect:** 536 entries — train 271, not_in_split 182, test 44, val 39.

In [25]:
bad = full[full["bulk_modulus_kv"] <= 0]
print(f"materials with an unphysical bulk modulus: {len(bad)}")
print(bad["split"].value_counts())

materials with an unphysical bulk modulus: 536
split
train           271
not_in_split    182
test             44
val              39
Name: count, dtype: int64


## 6. Suspicious values below 5 GPa

Values between 0 and 5 GPa are suspicious but **not** impossible — a few genuinely soft solids, such as
caesium, fall in this range. The paper therefore removes only values at or below zero, and reports this
wider count for context only.

**Expect:** 1,027 entries below 5 GPa.

In [26]:
suspicious = full[full["bulk_modulus_kv"] < 5]
print(f"Below 5 GPa: {len(suspicious)}")
print(suspicious["split"].value_counts())
print(suspicious.sort_values("bulk_modulus_kv").head(20)[["jid","formula","bulk_modulus_kv","split"]])

Below 5 GPa: 1027
split
train           596
not_in_split    288
test             74
val              69
Name: count, dtype: int64
                jid   formula  bulk_modulus_kv         split
60800  JVASP-137056     Al2Se           -24.59  not_in_split
76858  JVASP-161117   DyHoCo2           -24.33  not_in_split
50283  JVASP-115178      LiB2           -22.71         train
61240  JVASP-138670     SbSCl           -20.50  not_in_split
20976  JVASP-102481   Dy2ZnAg           -20.28         train
35491   JVASP-26529      CrF2           -20.08         train
62130  JVASP-138177    BeSbPb           -19.61  not_in_split
61398  JVASP-136827      SbSI           -19.13  not_in_split
37125   JVASP-31892     MoCl5           -17.10          test
52477  JVASP-117884      CSCl           -16.29         train
39933   JVASP-11470   Mn2CdO4           -16.18         train
62931  JVASP-136307     CdSCl           -16.06  not_in_split
48769  JVASP-114289     GeH2O           -15.99         train
64844  JVASP-134

## 7. Rate per split

Counts alone mislead, because the groups differ in size. Dividing by group size gives the rate.

**Expect:** between 1.4% and 2.2% in every group. Because the rate is similar everywhere, the problem runs
through the whole database rather than affecting one split — so the paper does **not** claim the test set
was unfairly corrupted.

In [28]:
rates = full[full["bulk_modulus_kv"] <= 0]["split"].value_counts()/ full["split"].value_counts()
print((rates*100).round(2))

split
train           1.72
not_in_split    1.43
test            2.24
val             1.98
Name: count, dtype: float64


## 8. Save

Writes the full table, with split labels, for later checks.

In [ ]:
import os
os.makedirs("../data", exist_ok=True)
full.to_csv("../data/full_bulk_modulus_audit.csv", index=False)